# Q-Shield: Siamese Network Training for QR Quishing Detection

**Phase 1:** Contrastive pretraining — teach the network which QR codes are similar/different  
**Phase 2:** Supervised fine-tuning — binary classifier on learned embeddings  

---
**Author:** Nicolas A. Llerena Silva (UTEC)  
**Advisor:** Aurea Soriano-Vargas  
**Backbone:** MobileNetV2 (2.9M params, shared weights)  
**Loss:** Contrastive Loss (Chopra et al., 2005)  
**Environment:** Google Colab Pro+ (T4/A100 GPU)

### Google Drive Structure
```
MyDrive/QShield/
  QuishingDataset.zip          <- Trad (9 MB)
  QR_benign_430K.zip           <- CIC benign QRs (241 MB)
  QR_malicious_576K.zip        <- CIC malicious QRs (319 MB)
```

In [ ]:
# ============================================================
# 0. MOUNT DRIVE & VERIFY STRUCTURE
# ============================================================
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/QShield'
else:
    BASE = '.'

# Verify Drive structure
print(f'Base: {BASE}')
print(f'Contents:')
for f in sorted(os.listdir(BASE)):
    size = os.path.getsize(os.path.join(BASE, f))
    print(f'  {f:<40} {size/1024/1024:.1f} MB')

# Check required files
required = ['QuishingDataset.zip']
optional = ['QR_benign_430K.zip', 'QR_malicious_576K.zip']
for f in required:
    path = os.path.join(BASE, f)
    assert os.path.exists(path), f'MISSING REQUIRED: {f}'
    print(f'\n[OK] {f}')

HAS_CIC = all(os.path.exists(os.path.join(BASE, f)) for f in optional)
print(f'\nCIC full dataset available: {HAS_CIC}')
if HAS_CIC:
    print('Training will use BOTH Trad (9,987) + CIC (1M+) datasets')
else:
    print('Training on Trad dataset only (still valid for paper)')

In [ ]:
# ============================================================
# 0.1 GPU CHECK & DEPENDENCIES
# ============================================================
if IN_COLAB:
    !nvidia-smi

!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ============================================================
# 0.2 IMPORTS
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models, transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, roc_curve, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from PIL import Image
from pathlib import Path
import pickle, zipfile, random, time, copy, json, glob
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('All imports ready.')

---
## 1. EXTRACT DATASETS FROM ZIPS

In [ ]:
# ============================================================
# 1.1 EXTRACT TRAD DATASET (9 MB -> 9,987 QR arrays)
# ============================================================
WORK = '/content/qshield_data'  # local SSD, much faster than Drive
os.makedirs(WORK, exist_ok=True)

trad_dir = os.path.join(WORK, 'trad')
if not os.path.exists(os.path.join(trad_dir, 'qr_codes_29.pickle')):
    os.makedirs(trad_dir, exist_ok=True)
    print('Extracting Trad dataset...')
    with zipfile.ZipFile(os.path.join(BASE, 'QuishingDataset.zip'), 'r') as z:
        z.extractall(trad_dir)
    print('Done.')

with open(os.path.join(trad_dir, 'qr_codes_29.pickle'), 'rb') as f:
    trad_qr = pickle.load(f)
with open(os.path.join(trad_dir, 'qr_codes_29_labels.pickle'), 'rb') as f:
    trad_labels = pickle.load(f)

print(f'Trad: {trad_qr.shape} ({trad_qr.dtype})')
print(f'  Benign:   {(trad_labels==0).sum()}')
print(f'  Phishing: {(trad_labels==1).sum()}')

In [ ]:
# ============================================================
# 1.2 EXTRACT CIC QR IMAGES (560 MB -> 1M+ PNGs)
# ============================================================
cic_benign_dir = os.path.join(WORK, 'cic_benign')
cic_malicious_dir = os.path.join(WORK, 'cic_malicious')

if HAS_CIC:
    # Extract benign
    if not os.path.exists(cic_benign_dir) or len(os.listdir(cic_benign_dir)) < 100:
        os.makedirs(cic_benign_dir, exist_ok=True)
        print('Extracting CIC benign QRs (430K images)...')
        with zipfile.ZipFile(os.path.join(BASE, 'QR_benign_430K.zip'), 'r') as z:
            z.extractall(cic_benign_dir)
        print('Done.')

    # Extract malicious
    if not os.path.exists(cic_malicious_dir) or len(os.listdir(cic_malicious_dir)) < 100:
        os.makedirs(cic_malicious_dir, exist_ok=True)
        print('Extracting CIC malicious QRs (576K images)...')
        with zipfile.ZipFile(os.path.join(BASE, 'QR_malicious_576K.zip'), 'r') as z:
            z.extractall(cic_malicious_dir)
        print('Done.')

    # Find all PNGs (may be in subdirectories)
    cic_benign_files = sorted(glob.glob(os.path.join(cic_benign_dir, '**', '*.png'), recursive=True))
    cic_malicious_files = sorted(glob.glob(os.path.join(cic_malicious_dir, '**', '*.png'), recursive=True))
    print(f'\nCIC benign:    {len(cic_benign_files):>8,} images')
    print(f'CIC malicious: {len(cic_malicious_files):>8,} images')
    print(f'CIC total:     {len(cic_benign_files)+len(cic_malicious_files):>8,} images')

    # Sample for training (use 10K per class for speed on first run)
    CIC_SAMPLE = 10000
    if len(cic_benign_files) > CIC_SAMPLE:
        cic_benign_files = random.sample(cic_benign_files, CIC_SAMPLE)
    if len(cic_malicious_files) > CIC_SAMPLE:
        cic_malicious_files = random.sample(cic_malicious_files, CIC_SAMPLE)
    print(f'\nSampled for training: {len(cic_benign_files)} benign + {len(cic_malicious_files)} malicious')
else:
    cic_benign_files = []
    cic_malicious_files = []
    print('CIC not available — using Trad only')

---
## 2. SIAMESE NETWORK ARCHITECTURE

**Advisor requirement:** Siamese Network with contrastive learning to learn embeddings  
that differentiate benign from malicious QR codes based on structural similarity.

In [ ]:
# ============================================================
# 2.1 MOBILENETV2 EMBEDDING BACKBONE
# ============================================================

class MobileNetV2Embedding(nn.Module):
    """
    Shared backbone for the Siamese Network.
    MobileNetV2 (ImageNet pretrained) -> 128-d L2-normalized embedding.
    Modified for single-channel grayscale input (QR codes).
    """

    def __init__(self, embedding_dim=128, pretrained=True):
        super().__init__()
        mobilenet = models.mobilenet_v2(
            weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        )
        original_conv = mobilenet.features[0][0]
        self.features = mobilenet.features
        # Adapt for 1-channel grayscale input
        self.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        if pretrained:
            with torch.no_grad():
                self.features[0][0].weight = nn.Parameter(
                    original_conv.weight.mean(dim=1, keepdim=True)
                )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Linear(1280, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embedding_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.projection(x)
        return F.normalize(x, p=2, dim=1)


class SiameseQRNet(nn.Module):
    """
    Siamese Network: two inputs share the SAME backbone.
    Learns a metric space where same-class QR codes are close
    and different-class QR codes are far apart.
    """

    def __init__(self, embedding_dim=128, pretrained=True):
        super().__init__()
        self.backbone = MobileNetV2Embedding(embedding_dim, pretrained)

    def forward_one(self, x):
        return self.backbone(x)

    def forward(self, x1, x2):
        return self.backbone(x1), self.backbone(x2)


class ContrastiveLoss(nn.Module):
    """
    Contrastive Loss (Chopra, Hadsell, LeCun 2005).
    Same-class pairs: minimize distance.
    Different-class pairs: push apart beyond margin.
    """

    def __init__(self, margin=2.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        dist = F.pairwise_distance(emb1, emb2)
        loss = (1 - label) * 0.5 * dist.pow(2) + \
               label * 0.5 * F.relu(self.margin - dist).pow(2)
        return loss.mean()


# Instantiate and check
model = SiameseQRNet(embedding_dim=128, pretrained=True).to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Siamese Network: {params:,} parameters')
print(f'On device: {device}')

---
## 3. PAIR DATASETS FOR CONTRASTIVE LEARNING

In [ ]:
# ============================================================
# 3.1 DATASET CLASSES
# ============================================================

class TradPairDataset(Dataset):
    """Pair dataset from Trad 69x69 binary matrices."""

    def __init__(self, qr_arrays, labels, pairs_per_epoch=20000):
        self.qr = qr_arrays.astype(np.float32)
        self.labels = np.array(labels)
        self.n = pairs_per_epoch
        self.idx = {0: np.where(self.labels == 0)[0],
                    1: np.where(self.labels == 1)[0]}

    def __len__(self):
        return self.n

    def _tensor(self, i):
        t = torch.from_numpy(self.qr[i]).unsqueeze(0).unsqueeze(0)
        return F.interpolate(t, size=(224, 224), mode='bilinear',
                             align_corners=False).squeeze(0)

    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0, 1])
        i1 = random.choice(self.idx[c])
        c2 = c if same else 1 - c
        i2 = random.choice(self.idx[c2])
        return self._tensor(i1), self._tensor(i2), torch.tensor(0.0 if same else 1.0)


class CICPairDataset(Dataset):
    """Pair dataset from CIC PNG images."""

    def __init__(self, benign_files, malicious_files, pairs_per_epoch=20000):
        self.files = {0: benign_files, 1: malicious_files}
        self.n = pairs_per_epoch

    def __len__(self):
        return self.n

    def _load(self, cls, idx):
        img = Image.open(self.files[cls][idx]).convert('L').resize((224, 224))
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).unsqueeze(0)

    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0, 1])
        i1 = random.randint(0, len(self.files[c]) - 1)
        c2 = c if same else 1 - c
        i2 = random.randint(0, len(self.files[c2]) - 1)
        return self._load(c, i1), self._load(c2, i2), torch.tensor(0.0 if same else 1.0)


class TradClassifyDataset(Dataset):
    """Single-image dataset for Phase 2 classification."""

    def __init__(self, qr_arrays, labels):
        self.qr = qr_arrays.astype(np.float32)
        self.labels = np.array(labels, dtype=np.float32)

    def __len__(self):
        return len(self.qr)

    def __getitem__(self, idx):
        t = torch.from_numpy(self.qr[idx]).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=(224, 224), mode='bilinear',
                          align_corners=False).squeeze(0)
        return t, torch.tensor(self.labels[idx])


print('Dataset classes defined.')

In [ ]:
# ============================================================
# 3.2 BUILD TRAIN/VAL SPLITS & DATALOADERS
# ============================================================
# Trad: 80/20 stratified split
idx_tr, idx_val = train_test_split(
    np.arange(len(trad_labels)), test_size=0.2,
    stratify=trad_labels, random_state=SEED
)

qr_tr, lab_tr = trad_qr[idx_tr], trad_labels[idx_tr]
qr_val, lab_val = trad_qr[idx_val], trad_labels[idx_val]

print(f'Trad train: {len(idx_tr)} | Val: {len(idx_val)}')

# Pair counts scale with data availability
BATCH = 64 if device.type == 'cuda' else 16
PAIRS_TR = 30000
PAIRS_VAL = 5000

# Build pair datasets
trad_pair_tr = TradPairDataset(qr_tr, lab_tr, PAIRS_TR)
trad_pair_val = TradPairDataset(qr_val, lab_val, PAIRS_VAL)

if HAS_CIC:
    # Split CIC files 80/20
    split = int(len(cic_benign_files) * 0.8)
    cic_pair_tr = CICPairDataset(cic_benign_files[:split],
                                  cic_malicious_files[:split], PAIRS_TR)
    cic_pair_val = CICPairDataset(cic_benign_files[split:],
                                   cic_malicious_files[split:], PAIRS_VAL)
    # Combine Trad + CIC
    train_ds = ConcatDataset([trad_pair_tr, cic_pair_tr])
    val_ds = ConcatDataset([trad_pair_val, cic_pair_val])
    print(f'Combined: {len(train_ds)} train pairs, {len(val_ds)} val pairs')
else:
    train_ds = trad_pair_tr
    val_ds = trad_pair_val

NUM_WORKERS = 4 if IN_COLAB else 0
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f'Batch size: {BATCH}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

---
## 4. PHASE 1: CONTRASTIVE PRETRAINING

The Siamese backbone learns: *"these two QR codes are structurally similar"* vs.  
*"these two QR codes are structurally different — one is benign, one is malicious."*

In [ ]:
# ============================================================
# 4.1 TRAINING CONFIG
# ============================================================
EMB_DIM = 128
MARGIN = 2.0
LR1 = 1e-4
EPOCHS1 = 30

model = SiameseQRNet(embedding_dim=EMB_DIM, pretrained=True).to(device)
criterion = ContrastiveLoss(margin=MARGIN)
optimizer = optim.AdamW(model.parameters(), lr=LR1, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS1, eta_min=1e-6)

print(f'Phase 1: {EPOCHS1} epochs, LR={LR1}, margin={MARGIN}')
print(f'Pairs/epoch: {len(train_ds)} train, {len(val_ds)} val')

In [ ]:
# ============================================================
# 4.2 TRAINING LOOP
# ============================================================
hist1 = {'tl': [], 'vl': [], 'ta': [], 'va': []}
best_vl = float('inf')
best_state = None

print(f'{"Ep":>3} {"TrLoss":>8} {"VaLoss":>8} {"TrAcc":>7} {"VaAcc":>7} {"LR":>10} {"Time":>6}')
print('-' * 58)

for ep in range(1, EPOCHS1 + 1):
    t0 = time.time()

    # Train
    model.train()
    tl, tc, tt = 0, 0, 0
    for x1, x2, y in train_loader:
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)
        optimizer.zero_grad()
        e1, e2 = model(x1, x2)
        loss = criterion(e1, e2, y)
        loss.backward()
        optimizer.step()
        tl += loss.item() * x1.size(0)
        with torch.no_grad():
            d = F.pairwise_distance(e1, e2)
            tc += ((d > MARGIN/2).float() == y).sum().item()
            tt += y.size(0)

    # Val
    model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for x1, x2, y in val_loader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            e1, e2 = model(x1, x2)
            loss = criterion(e1, e2, y)
            vl += loss.item() * x1.size(0)
            d = F.pairwise_distance(e1, e2)
            vc += ((d > MARGIN/2).float() == y).sum().item()
            vt += y.size(0)

    scheduler.step()
    tl_avg, vl_avg = tl/tt, vl/vt
    ta, va = tc/tt, vc/vt
    hist1['tl'].append(tl_avg)
    hist1['vl'].append(vl_avg)
    hist1['ta'].append(ta)
    hist1['va'].append(va)

    mk = ''
    if vl_avg < best_vl:
        best_vl = vl_avg
        best_state = copy.deepcopy(model.state_dict())
        mk = ' *'

    lr = optimizer.param_groups[0]['lr']
    dt = time.time() - t0
    print(f'{ep:>3} {tl_avg:>8.4f} {vl_avg:>8.4f} {ta:>6.1%} {va:>6.1%} {lr:>10.6f} {dt:>5.0f}s{mk}')

model.load_state_dict(best_state)
torch.save(best_state, os.path.join(BASE, 'siamese_phase1.pth'))
print(f'\nBest val loss: {best_vl:.4f} — saved to Drive')

In [ ]:
# ============================================================
# 4.3 TRAINING CURVES
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist1['tl'], label='Train', lw=2)
axes[0].plot(hist1['vl'], label='Val', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Contrastive Loss')
axes[0].set_title('Phase 1: Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist1['ta'], label='Train', lw=2)
axes[1].plot(hist1['va'], label='Val', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Pair Accuracy')
axes[1].set_title('Phase 1: Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle('Siamese Contrastive Pretraining', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_phase1_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5. EMBEDDING VISUALIZATION (t-SNE)

If contrastive learning worked, benign and phishing QR codes should form  
**separable clusters** in the embedding space.

In [ ]:
# ============================================================
# 5.1 EXTRACT & VISUALIZE EMBEDDINGS
# ============================================================
val_ds_cls = TradClassifyDataset(qr_val, lab_val)
val_cls_loader = DataLoader(val_ds_cls, batch_size=128, shuffle=False, num_workers=NUM_WORKERS)

model.eval()
embs, labs = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(val_cls_loader, desc='Embeddings'):
        embs.append(model.forward_one(imgs.to(device)).cpu().numpy())
        labs.append(lbls.numpy())
embs = np.concatenate(embs)
labs = np.concatenate(labs)

print(f'Embeddings: {embs.shape}')
print('Running t-SNE...')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
proj = tsne.fit_transform(embs)

fig, ax = plt.subplots(figsize=(8, 7))
for lbl, color, name in [(0, '#2ecc71', 'Benign'), (1, '#e74c3c', 'Phishing')]:
    m = labs == lbl
    ax.scatter(proj[m, 0], proj[m, 1], c=color, label=name, alpha=0.5, s=12, edgecolors='none')
ax.set_title('Siamese Embedding Space (t-SNE) — Trad Val Set', fontweight='bold')
ax.legend(markerscale=3); ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_tsne_embeddings.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Separable clusters = contrastive learning succeeded.')

---
## 6. PHASE 2: SUPERVISED CLASSIFICATION

Attach a classification head to the pretrained backbone.  
Fine-tune end-to-end with binary cross-entropy.

In [ ]:
# ============================================================
# 6.1 CLASSIFIER HEAD
# ============================================================

class QRClassifier(nn.Module):
    def __init__(self, backbone, emb_dim=128):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(emb_dim, 256), nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


classifier = QRClassifier(model.backbone, EMB_DIM).to(device)
print(f'Classifier params: {sum(p.numel() for p in classifier.parameters() if p.requires_grad):,}')

In [ ]:
# ============================================================
# 6.2 PHASE 2 TRAINING
# ============================================================
EPOCHS2 = 20
LR2 = 5e-5

tr_cls_loader = DataLoader(TradClassifyDataset(qr_tr, lab_tr),
                           batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
va_cls_loader = DataLoader(TradClassifyDataset(qr_val, lab_val),
                           batch_size=128, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

bce = nn.BCEWithLogitsLoss()
opt2 = optim.AdamW(classifier.parameters(), lr=LR2, weight_decay=1e-4)
sch2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS2, eta_min=1e-6)

hist2 = {'tl': [], 'vl': [], 'auc': [], 'f1': []}
best_auc = 0
best_cls = None

print(f'{"Ep":>3} {"TrLoss":>8} {"VaLoss":>8} {"AUC":>7} {"F1":>7} {"Prec":>7} {"Rec":>7}')
print('-' * 56)

for ep in range(1, EPOCHS2 + 1):
    classifier.train()
    tl = 0
    for imgs, lbls in tr_cls_loader:
        imgs, lbls = imgs.to(device), lbls.to(device).unsqueeze(1)
        opt2.zero_grad()
        loss = bce(classifier(imgs), lbls)
        loss.backward()
        opt2.step()
        tl += loss.item() * imgs.size(0)

    classifier.eval()
    vl = 0; probs_all, true_all = [], []
    with torch.no_grad():
        for imgs, lbls in va_cls_loader:
            imgs, lbls = imgs.to(device), lbls.to(device).unsqueeze(1)
            logits = classifier(imgs)
            vl += bce(logits, lbls).item() * imgs.size(0)
            probs_all.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            true_all.extend(lbls.cpu().numpy().flatten())

    sch2.step()
    probs_all = np.array(probs_all); true_all = np.array(true_all)
    preds = (probs_all >= 0.5).astype(int)
    auc = roc_auc_score(true_all, probs_all)
    f1 = f1_score(true_all, preds)
    prec = precision_score(true_all, preds)
    rec = recall_score(true_all, preds)

    hist2['tl'].append(tl/len(qr_tr)); hist2['vl'].append(vl/len(qr_val))
    hist2['auc'].append(auc); hist2['f1'].append(f1)

    mk = ''
    if auc > best_auc:
        best_auc = auc
        best_cls = copy.deepcopy(classifier.state_dict())
        mk = ' *'

    print(f'{ep:>3} {tl/len(qr_tr):>8.4f} {vl/len(qr_val):>8.4f} {auc:>6.4f} {f1:>6.4f} {prec:>6.4f} {rec:>6.4f}{mk}')

classifier.load_state_dict(best_cls)
torch.save(best_cls, os.path.join(BASE, 'classifier_phase2.pth'))
print(f'\nBest AUC: {best_auc:.4f} — saved to Drive')

---
## 7. FINAL EVALUATION & PAPER FIGURES

In [ ]:
# ============================================================
# 7.1 FINAL METRICS + ROC + CONFUSION MATRIX
# ============================================================
classifier.eval()
probs_all, true_all = [], []
with torch.no_grad():
    for imgs, lbls in va_cls_loader:
        logits = classifier(imgs.to(device))
        probs_all.extend(torch.sigmoid(logits).cpu().numpy().flatten())
        true_all.extend(lbls.numpy().flatten())

probs_all = np.array(probs_all); true_all = np.array(true_all)
preds = (probs_all >= 0.5).astype(int)
final_auc = roc_auc_score(true_all, probs_all)
final_f1 = f1_score(true_all, preds)

print('='*60)
print(' FINAL RESULTS — Siamese + Classifier (Trad Val Set)')
print('='*60)
print(f'AUC: {final_auc:.4f}')
print(classification_report(true_all, preds, target_names=['Benign', 'Phishing']))

# Plots
cm = confusion_matrix(true_all, preds)
fpr, tpr, _ = roc_curve(true_all, probs_all)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Benign', 'Phishing'], yticklabels=['Benign', 'Phishing'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (AUC={final_auc:.4f})', fontweight='bold')

axes[1].plot(fpr, tpr, lw=2, color='#e74c3c', label=f'Siamese (AUC={final_auc:.4f})')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('Q-Shield Siamese Classifier — Final Evaluation', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_final_results.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 7.2 COMPARISON TABLE & SAVE RESULTS
# ============================================================
print('\nMETHOD COMPARISON — Trad Dataset')
print('='*70)
comp = pd.DataFrame([
    {'Method': 'Trad et al. (4761 raw pixels + XGBoost)', 'AUC': 0.9133, 'Params': '4,761 feat', 'Explainable': 'Low'},
    {'Method': 'Ours: 25 handcrafted + RF', 'AUC': 0.8132, 'Params': '25 feat', 'Explainable': 'High (SHAP)'},
    {'Method': 'Ours: Siamese MobileNetV2', 'AUC': round(final_auc, 4), 'Params': '2.9M', 'Explainable': 'Med (Grad-CAM)'},
])
print(comp.to_string(index=False))

results = {
    'phase1': hist1, 'phase2': hist2,
    'final_auc': final_auc, 'final_f1': final_f1,
    'confusion_matrix': cm.tolist(),
    'config': {
        'embedding_dim': EMB_DIM, 'margin': MARGIN,
        'lr_phase1': LR1, 'lr_phase2': LR2,
        'epochs_phase1': EPOCHS1, 'epochs_phase2': EPOCHS2,
        'batch_size': BATCH, 'has_cic': HAS_CIC,
    }
}
with open(os.path.join(BASE, 'experiment_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nAll results saved to {BASE}/')
for fn in sorted(os.listdir(BASE)):
    if not fn.startswith('.') and os.path.isfile(os.path.join(BASE, fn)):
        sz = os.path.getsize(os.path.join(BASE, fn)) / 1024
        print(f'  {fn:<40} {sz:>8.0f} KB')